In [0]:
# OPTIMIZE compacts small files into larger ones
# This improves query performance significantly

print("Optimizing Bronze Tables...")
spark.sql("OPTIMIZE bronze_db.bronze_financials")
spark.sql("OPTIMIZE bronze_db.bronze_failed_banks")
spark.sql("OPTIMIZE bronze_db.bronze_institutions")
spark.sql("OPTIMIZE bronze_db.bronze_summary")
print("Bronze tables optimized!")

print("Optimizing Silver Tables...")
spark.sql("OPTIMIZE silver_db.silver_bank_features")
print("Silver tables optimized!")

print("Optimizing Gold Tables...")
spark.sql("OPTIMIZE gold_db.gold_bank_features")
spark.sql("OPTIMIZE gold_db.gold_predictions")
spark.sql("OPTIMIZE gold_db.gold_inference_results")
spark.sql("OPTIMIZE gold_db.gold_analytics_summary")
print("Gold tables optimized!")

Optimizing Bronze Tables...
Bronze tables optimized!
Optimizing Silver Tables...
Silver tables optimized!
Optimizing Gold Tables...
Gold tables optimized!


In [0]:
# ZORDER co-locates related data in same files
# Makes queries on these columns much faster

print("Applying ZORDER...")

# Most queried column in financials is CERT
spark.sql("OPTIMIZE bronze_db.bronze_financials ZORDER BY (CERT)")

# Most queried column in failed banks is CERT and FAILDATE
spark.sql("OPTIMIZE bronze_db.bronze_failed_banks ZORDER BY (CERT)")

# Most queried in silver is CERT and failed label
spark.sql("OPTIMIZE silver_db.silver_bank_features ZORDER BY (CERT)")

# Most queried in gold is failure_probability and contagion_risk
spark.sql("OPTIMIZE gold_db.gold_inference_results ZORDER BY (failure_probability)")

print("ZORDER applied!")

Applying ZORDER...
ZORDER applied!


In [0]:
# Time travel lets you query older versions of data
# This is a key Delta Lake feature

# Check history of gold_inference_results
print("Delta Table History:")
display(spark.sql("DESCRIBE HISTORY gold_db.gold_inference_results"))

Delta Table History:


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-03-13T17:20:12.000Z,72454969090887,margithakar851@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4103136464093466),ea659e0a-4b37-4cee-a337-ee237d30e732,0313-150727-bf2ctfd0-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 108, numOutputBytes -> 8553)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13


In [0]:
# Query previous version of the table (time travel)
print("Current version:")
print(spark.table("gold_db.gold_inference_results").count(), "rows")

print("\nVersion 0 (first version):")
print(spark.sql("SELECT * FROM gold_db.gold_inference_results VERSION AS OF 0").count(), "rows")

Current version:
108 rows

Version 0 (first version):
108 rows


In [0]:
spark.sql("VACUUM bronze_db.bronze_financials")
spark.sql("VACUUM silver_db.silver_bank_features")
spark.sql("VACUUM gold_db.gold_inference_results")

print("VACUUM complete — old files cleaned up!")

VACUUM complete — old files cleaned up!


In [0]:
# Verify schema enforcement is working
print("Bronze Financials Schema:")
spark.table("bronze_db.bronze_financials").printSchema()

print("Silver Bank Features Schema:")
spark.table("silver_db.silver_bank_features").printSchema()

print("Gold Inference Results Schema:")
spark.table("gold_db.gold_inference_results").printSchema()

Bronze Financials Schema:
root
 |-- REPDTE: string (nullable = true)
 |-- ASSET: long (nullable = true)
 |-- CERT: long (nullable = true)
 |-- NETINC: long (nullable = true)
 |-- LNLSNET: long (nullable = true)
 |-- DEP: long (nullable = true)
 |-- EQTOT: long (nullable = true)
 |-- ID: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source: string (nullable = true)
 |-- layer: string (nullable = true)

Silver Bank Features Schema:
root
 |-- CERT: integer (nullable = true)
 |-- REPDTE: string (nullable = true)
 |-- ASSET: double (nullable = true)
 |-- NETINC: double (nullable = true)
 |-- LNLSNET: double (nullable = true)
 |-- DEP: double (nullable = true)
 |-- EQTOT: double (nullable = true)
 |-- ID: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source: string (nullable = true)
 |-- layer: string (nullable = true)
 |-- capital_ratio: double (nullable = true)
 |-- loan_to_deposit_ratio: double (nullable = true)
 